<a href="https://colab.research.google.com/github/satani99/triton_kernels/blob/main/profiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

In [2]:
a = torch.tensor([1., 2., 3.])

In [3]:
print(torch.square(a))
print(a ** 2)
print(a * a)

tensor([1., 4., 9.])
tensor([1., 4., 9.])
tensor([1., 4., 9.])


In [4]:
def time_pytorch_function(func, input):
  start = torch.cuda.Event(enable_timing=True)
  end = torch.cuda.Event(enable_timing=True)

  for _ in range(5):
    func(input)

  start.record()
  func(input)
  end.record()
  torch.cuda.synchronize()
  return start.elapsed_time(end)

In [5]:
b = torch.randn(10000, 10000).cuda()

In [6]:
def square_2(a):
  return a * a

In [7]:
def square_3(a):
  return a ** 2

In [8]:
time_pytorch_function(torch.square, b)


3.3519039154052734

In [9]:
time_pytorch_function(square_2, b)


3.3443520069122314

In [10]:
time_pytorch_function(square_3, b)

3.3447999954223633

In [11]:
print("============")
print("Profiling torch.square")
print("============")

with torch.profiler.profile() as prof:
  torch.square(b)

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))

Profiling torch.square
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                           aten::square         0.18%       9.583us        40.38%       2.174ms       2.174ms       0.000us         0.00%       6.654ms       6.654ms             1  
                                              aten::pow         1.76%      94.621us        40.21%       2.164ms       2.164ms       3.327ms       100.00%       6.654ms       6.654ms   

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


In [12]:
print("============")
print("Profiling a * a")
print("============")

with torch.profiler.profile() as prof:
  square_2(b)

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))

Profiling a * a
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                              aten::mul         1.53%      89.653us        44.71%       2.626ms       2.626ms       3.340ms       100.00%       6.680ms       6.680ms             1  
                                Activity Buffer Request        42.09%       2.472ms        42.09%       2.472ms       2.472ms       3.340ms       100.00%       3.340ms       3.340ms          

In [13]:
print("===========")
print("Profiling a ** 2")
print("===========")

with torch.profiler.profile() as prof:
  square_3(b)

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))

Profiling a ** 2
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                              aten::pow         1.61%      93.558us        43.75%       2.541ms       2.541ms       3.337ms       100.00%       6.674ms       6.674ms             1  
                                Activity Buffer Request        41.37%       2.403ms        41.37%       2.403ms       2.403ms       3.337ms       100.00%       3.337ms       3.337ms         